# 03 — Baselines

This notebook documents the majority-class and handcrafted-feature logistic-regression references on the frozen **Original** benchmark. Feature extraction, metric computation, standardization, and logistic fitting live in reusable source modules; the notebook reads the saved outputs and independently recomputes their reported metrics.


In [ ]:
from pathlib import Path
import csv
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError("Run from the repository root or notebooks/ directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation import compute_binary_metrics
from src.features import FEATURE_SETS, FEATURE_SIZE

SUMMARY_PATH = ROOT / "outputs" / "metrics" / "baseline_summary.json"
TABLES = ROOT / "outputs" / "tables"
PREDICTIONS = ROOT / "outputs" / "predictions"
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))


## Handcrafted feature design

All feature sets are nested. Set A contains brightness, contrast, sharpness, entropy, and edge density. Set B adds RGB statistics, saturation/hue information, and grayscale intensity quantiles. Set C adds grayscale histograms, local-binary-pattern texture bins, and gradient-orientation structure features. Features are extracted from a fixed 224 × 398 RGB representation for a controlled, inexpensive baseline—not as a replacement for the later image models.


In [ ]:
print("feature resolution:", FEATURE_SIZE)
for name in ("A", "B", "C"):
    print(f"Set {name}: {len(FEATURE_SETS[name])} features")
print("feature table sha256:", summary["feature_csv"]["sha256"])


## Tuning protocol

The supplied test set remains untouched during model selection. Inside the supplied training pool, a deterministic seed-42 stratified 90/10 subdivision provides fit and validation populations. Each feature set independently searches L2-regularized logistic `C` from `1e-4` through `1e4`. Standardization is fit on the fit population only during selection. After selecting the best feature set and `C` by validation F1, the chosen baseline is refit on the complete supplied training set and evaluated once on the supplied test set. The classification threshold stays fixed at 0.5.


In [ ]:
print(json.dumps(summary["tuning"], indent=2))
search_rows = list(csv.DictReader((TABLES / "logistic_c_search.csv").open(newline="", encoding="utf-8")))
for set_name in ("A", "B", "C"):
    rows = [row for row in search_rows if row["feature_set"] == set_name]
    best = max(rows, key=lambda row: float(row["f1"]))
    print(f"Set {set_name}: best validation F1={float(best['f1']):.4f} at C={best['C']}")


## Baseline results

`flip` is the positive class. The majority model is intentionally minimal: its probability is the positive-class prevalence estimated from training only, with the same fixed 0.5 decision threshold used by every later model. The handcrafted logistic baseline uses native logistic probabilities.


In [ ]:
print("Majority — validation")
print(json.dumps(summary["majority"]["validation_metrics"], indent=2, sort_keys=True))
print("\nMajority — supplied test")
print(json.dumps(summary["majority"]["test_metrics"], indent=2, sort_keys=True))
print("\nSelected handcrafted logistic")
print("feature set:", summary["handcrafted_logistic"]["selected_feature_set"])
print("C:", summary["handcrafted_logistic"]["selected_C"])
print("validation:")
print(json.dumps(summary["handcrafted_logistic"]["validation_metrics"], indent=2, sort_keys=True))
print("test:")
print(json.dumps(summary["handcrafted_logistic"]["test_metrics"], indent=2, sort_keys=True))


## Prediction-output verification

The metrics below are recomputed from the saved prediction CSVs rather than copied from the training routine. This is the Phase-4 reproducibility gate for the baseline numbers.


In [ ]:
def metrics_from_prediction_file(path):
    rows = list(csv.DictReader(path.open(newline="", encoding="utf-8")))
    truth = [int(row["true_class"]) for row in rows]
    probability = [float(row["probability_flip"]) for row in rows]
    return compute_binary_metrics(truth, probability)

majority_test_check = metrics_from_prediction_file(PREDICTIONS / "majority_original_test.csv")
logistic_test_check = metrics_from_prediction_file(PREDICTIONS / "handcrafted_logistic_original_test.csv")
assert majority_test_check == summary["majority"]["test_metrics"]
assert logistic_test_check == summary["handcrafted_logistic"]["test_metrics"]
print("PASS: saved predictions reproduce both supplied-test metric bundles exactly")


## Interpretable logistic coefficients

Coefficients are expressed in standardized feature space, so absolute magnitude provides a simple description of which handcrafted measurements the selected baseline relies on most strongly. This is descriptive, not causal.


In [ ]:
coefficients = list(csv.DictReader((TABLES / "handcrafted_logistic_coefficients.csv").open(newline="", encoding="utf-8")))
for row in coefficients[:12]:
    print(f"{row['feature']}: {float(row['standardized_coefficient']):+.4f}")


## Phase-4 conclusion

The baseline stage establishes two deliberately simple references before CNN tuning: an information-free majority classifier and a regularized linear classifier over interpretable visual statistics. Their role is comparative. No conclusion about the later scratch CNN or pretrained model is made here.
